# California Collision Severity Mining — Code Analysis & Optimized Pipeline

## Full Code Analysis of Previous `test.ipynb`

| Area | Issue Found in Previous Notebook | Optimization Applied Here |
| :--- | :--- | :--- |
| **Old Cells 1–5 (`process` loop)** | Called `SGDClassifier.fit()` from scratch on every 100k chunk (resetting weights every batch instead of `partial_fit`) and labeled it `Baseline Model (H0)`. Also set `echo=True`, flooding output with SQL logs over 96 batches (~43 min). | Removed redundant loop; replaced with clean 3-step pipeline (`echo=False`). |
| **DB Query & Streaming** | `pd.read_sql("SELECT * FROM mart.vw_raw_collision", engine, chunksize=...)` fetched all 50 columns (including large text/ID fields) without a server-side cursor (`stream_results=True`), increasing network I/O and client RAM buffering. | Project only the 22 valid features + 1 target in SQL and stream with `execution_options(stream_results=True)` inside a `with` context manager so the cursor closes immediately at 500k rows. |
| **Missing Value Filtering** | Calling `chunk.dropna()` on all 50 columns dropped valid highway crashes whenever unused columns (e.g., `secondary_road`) were `NULL`. | Apply `.dropna()` strictly on `FEATURES + [TARGET]`. |
| **Variable Scope & Reproducibility** | `FEATURES` was deleted from the loader cell, causing `NameError` on a fresh kernel restart. | Defined `TARGET`, `NUM_COLS`, `CAT_COLS`, and `FEATURES` explicitly in Cell 1. |
| **Memory & Speed** | Full float64/object DataFrames and re-encoding for each model wasted RAM and CPU time. | Downcast numerics to `float32`, encode once with sparse `OneHotEncoder`, and slice the encoded matrix directly (`[:, top30_idx]`) for the Top 30 feature models (~48 MB dense `float32`). |


---
## 1. Stream & Clean 500k Records from PostgreSQL (All Valid Features)
Streams from `mart.vw_raw_collision` in 100k chunks, drops NAs only on valid features + target, and stores 500k clean records in memory with low RAM overhead.

In [ ]:
import os, gc, time
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
t0 = time.perf_counter()

# 1. Connect to PostgreSQL (echo=False keeps output clean)
engine = create_engine(
    f"postgresql+psycopg://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT') or 5432}/{os.getenv('PGDATABASE')}",
    echo=False
)

# 2. Target & All 22 Valid Non-Leakage Features
TARGET = 'severe_or_fatal'
NUM_COLS = [
    'hour', 'minute_of_day', 'month', 'quarter', 'year', 'iso_weekday',
    'latitude', 'longitude', 'station_distance_m',
    'prcp_mm', 'tmin_c', 'tmax_c', 'population'
]
CAT_COLS = [
    'lighting', 'road_surface', 'reported_weather', 'trace_precipitation',
    'primary_collision_factor', 'type_of_collision',
    'county_name', 'time_band', 'is_weekend'
]
FEATURES = NUM_COLS + CAT_COLS

# 3. Stream only needed columns with a server-side cursor, drop NAs, collect 500k
TARGET_ROWS = 500_000
cols_sql = ', '.join(FEATURES + [TARGET])
query = f"SELECT {cols_sql} FROM mart.vw_raw_collision"

temp, total = [], 0
with engine.connect().execution_options(stream_results=True) as conn:
    for batch_idx, chunk in enumerate(pd.read_sql(query, conn, chunksize=100_000), 1):
        clean = chunk.dropna()
        if not clean.empty:
            temp.append(clean)
            total += len(clean)
        print(f"Batch {batch_idx:02d}: +{len(clean):,} valid rows (Total: {min(total, TARGET_ROWS):,} / {TARGET_ROWS:,})")
        if total >= TARGET_ROWS:
            break

df_clean = pd.concat(temp, ignore_index=True).iloc[:TARGET_ROWS].copy()
del temp
gc.collect()

# 4. Optimize memory dtypes
df_clean[NUM_COLS] = df_clean[NUM_COLS].astype(np.float32)
df_clean[CAT_COLS] = df_clean[CAT_COLS].astype(str)
df_clean[TARGET] = df_clean[TARGET].astype(np.int8)

ram_mb = df_clean.memory_usage(deep=True).sum() / 1e6
print(f"\nDone in {time.perf_counter() - t0:.1f}s | Shape: {df_clean.shape} | RAM: {ram_mb:.1f} MB | Severe rate: {df_clean[TARGET].mean()*100:.2f}%")


---
## 2. Base Model (All Features) & Top 30 Feature Importance
Encodes all 22 features once, compares the Base Decision Tree against the $H_0$ majority baseline, and extracts the **Top 30 most informative features**.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score, f1_score, roc_auc_score, roc_curve

# 1. Stratified 80/20 Train-Test Split
X = df_clean[FEATURES]
y = df_clean[TARGET].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Single Sparse Preprocessing Pass
preprocessor = make_column_transformer(
    (StandardScaler(), NUM_COLS),
    (OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.float32), CAT_COLS)
)
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)
feature_names = np.array([n.split('__')[-1] for n in preprocessor.get_feature_names_out()])
print(f"Encoded matrix: {X_train_enc.shape[0]:,} train rows x {X_train_enc.shape[1]} features")

# 3. H0 Null Baseline vs Base Model (All Features)
h0_model = DummyClassifier(strategy='most_frequent').fit(X_train_enc, y_train)
y_pred_h0 = h0_model.predict(X_test_enc)

t0 = time.perf_counter()
base_model = DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42)
base_model.fit(X_train_enc, y_train)
base_time = time.perf_counter() - t0

y_pred_base = base_model.predict(X_test_enc)
y_prob_base = base_model.predict_proba(X_test_enc)[:, 1]

print(f"H0 (Null Majority)         -> ROC-AUC: 0.5000 | Balanced Acc: {balanced_accuracy_score(y_test, y_pred_h0):.4f} | Recall: {recall_score(y_test, y_pred_h0, zero_division=0):.4f}")
print(f"Base Tree (All {X_train_enc.shape[1]} feats) -> ROC-AUC: {roc_auc_score(y_test, y_prob_base):.4f} | Balanced Acc: {balanced_accuracy_score(y_test, y_pred_base):.4f} | Recall: {recall_score(y_test, y_pred_base):.4f} ({base_time:.1f}s)")

# 4. Extract & Plot Top 30 Features
TOP_K = min(30, X_train_enc.shape[1])
top30_idx = np.argsort(base_model.feature_importances_)[::-1][:TOP_K]
top30_imp = pd.Series(base_model.feature_importances_[top30_idx], index=feature_names[top30_idx])

print(f"\n=== TOP {TOP_K} FEATURES (Base Model) ===")
print(top30_imp.round(5))

plt.figure(figsize=(10, 8))
top30_imp.plot(kind='barh', color='steelblue', edgecolor='black').invert_yaxis()
plt.title(f"Top {TOP_K} Feature Importances (Base Decision Tree — All Features)")
plt.xlabel("Gini Importance")
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


---
## 3. Train & Compare 3 Models on Top 30 Features
Slices the encoded matrix directly to the Top 30 features (`float32`, ~48 MB) and compares **Logistic Regression**, **Random Forest**, and **HistGradientBoosting** against the Base Model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# 1. Slice directly to Top 30 encoded features (compact float32 dense array ~48 MB)
to_dense = lambda m: (m.toarray() if hasattr(m, 'toarray') else np.asarray(m)).astype(np.float32)
X_train_top30 = to_dense(X_train_enc[:, top30_idx])
X_test_top30 = to_dense(X_test_enc[:, top30_idx])

# 2. Define 3 diverse models for Top 30 Features
top30_models = {
    "Logistic Regression (Top 30)": LogisticRegression(
        class_weight='balanced', max_iter=500, random_state=42
    ),
    "Random Forest (Top 30)": RandomForestClassifier(
        n_estimators=100, max_depth=12, min_samples_leaf=20,
        class_weight='balanced', n_jobs=-1, random_state=42
    ),
    "HistGradientBoosting (Top 30)": HistGradientBoostingClassifier(
        max_iter=150, max_depth=8, class_weight='balanced', random_state=42
    ),
}

# 3. Train, Evaluate & Collect ROC Curves
results = [
    {
        "Model": "H0 (Null Majority)",
        "Features": 0,
        "Accuracy": round(accuracy_score(y_test, y_pred_h0), 4),
        "Balanced Acc": round(balanced_accuracy_score(y_test, y_pred_h0), 4),
        "Recall": round(recall_score(y_test, y_pred_h0, zero_division=0), 4),
        "F1": round(f1_score(y_test, y_pred_h0, zero_division=0), 4),
        "ROC-AUC": 0.5000,
        "Train Sec": 0.0,
    },
    {
        "Model": "Base Decision Tree (All Features)",
        "Features": X_train_enc.shape[1],
        "Accuracy": round(accuracy_score(y_test, y_pred_base), 4),
        "Balanced Acc": round(balanced_accuracy_score(y_test, y_pred_base), 4),
        "Recall": round(recall_score(y_test, y_pred_base), 4),
        "F1": round(f1_score(y_test, y_pred_base), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_prob_base), 4),
        "Train Sec": round(base_time, 2),
    },
]

roc_curves = {"Base Decision Tree (All Features)": (y_prob_base, roc_auc_score(y_test, y_prob_base))}

for name, clf in top30_models.items():
    t0 = time.perf_counter()
    clf.fit(X_train_top30, y_train)
    elapsed = time.perf_counter() - t0

    preds = clf.predict(X_test_top30)
    probs = clf.predict_proba(X_test_top30)[:, 1]
    auc = roc_auc_score(y_test, probs)
    roc_curves[name] = (probs, auc)

    results.append({
        "Model": name,
        "Features": TOP_K,
        "Accuracy": round(accuracy_score(y_test, preds), 4),
        "Balanced Acc": round(balanced_accuracy_score(y_test, preds), 4),
        "Recall": round(recall_score(y_test, preds), 4),
        "F1": round(f1_score(y_test, preds), 4),
        "ROC-AUC": round(auc, 4),
        "Train Sec": round(elapsed, 2),
    })

results_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False).reset_index(drop=True)
print("=== MODEL COMPARISON (ALL FEATURES VS TOP 30 FEATURES) ===")
display(results_df)

# 4. Plot ROC Curves & Score Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: ROC Curves
axes[0].plot([0, 1], [0, 1], 'k--', label='H0 Random / Majority (AUC = 0.5000)')
for name, (probs, auc) in roc_curves.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    axes[0].plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc:.4f})")
axes[0].set_title("ROC Curves: All Features vs Top 30 Features")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate (Recall)")
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(alpha=0.3)

# Right: ROC-AUC & Balanced Accuracy Bar Chart
plot_df = results_df.set_index('Model')[['ROC-AUC', 'Balanced Acc', 'Recall']]
plot_df.plot(kind='barh', ax=axes[1], edgecolor='black')
axes[1].invert_yaxis()
axes[1].set_xlim(0, 1.0)
axes[1].set_title("Evaluation Metrics by Model")
axes[1].set_xlabel("Score")
axes[1].grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()
